# DMN Colab - By Ticker

Notebook Colab pour evaluer une strategie ticker par ticker via `python -m dmn.cli.by_ticker`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/figaroldamien/DMN.git"
REPO_DIR = Path("/content/DMN")
DRIVE_DIR = Path("/content/drive/MyDrive/DMN")
CONFIGS_DIR = DRIVE_DIR / "configs"
LOGS_DIR = DRIVE_DIR / "logs"

CONFIGS_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR / "notebooks"))
from notebook_logging import run_logged

print(f"Working directory: {REPO_DIR}")
print(f"Configs directory: {CONFIGS_DIR}")
print(f"Logs directory: {LOGS_DIR}")


In [ ]:
CONFIG_PATH = str(CONFIGS_DIR / "by_ticker.toml")
LOG_PATH = str(LOGS_DIR / "by_ticker.log")
USE_CONFIG_FILE = False
STRATEGY = "DMN_LSTM_Sharpe_TurnPen"
MARKET = "cac40"
TICKER = None
START = "2000-01-01"
SECTOR = None
SUB_SECTOR = None
PRINT_CONFIG = True


In [ ]:
env = os.environ.copy()
env["PYTHONPATH"] = f"{REPO_DIR / 'src'}:{env.get('PYTHONPATH', '')}"

cmd = [sys.executable, "-m", "dmn.cli.by_ticker", "--strategy", STRATEGY]
if USE_CONFIG_FILE:
    cmd.extend(["--config", CONFIG_PATH])
if TICKER:
    cmd.extend(["--ticker", TICKER])
else:
    cmd.extend(["--market", MARKET])
if START:
    cmd.extend(["--start", START])
if SECTOR:
    cmd.extend(["--sector", SECTOR])
if SUB_SECTOR:
    cmd.extend(["--sub-sector", SUB_SECTOR])
if not PRINT_CONFIG:
    cmd.append("--no-print-config")

if USE_CONFIG_FILE and not Path(CONFIG_PATH).exists():
    raise FileNotFoundError(f"Config file not found: {CONFIG_PATH}")

run_logged(cmd, env=env, log_path=LOG_PATH)
